# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AzlanFaisalRaj/flyrank-internship-ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding #1 — "The Anatomy of Growing Content" (page 6).** The paper compares pages with `trend_direction = up` against `trend_direction = down` and reports growing pages are 37.6% longer and 20% younger. My methodology question: where does `up`/`down` come from? It's a 30-day-vs-prior-30-day impression swing (>10% = up, >10% decline = down), which is the same short, noisy window my own `is_declining_label` is built from. A single 30-day swing can reflect a seasonal blip as easily as a real structural trend, so before I'd act on "younger and longer wins," I'd want to see the comparison held on a longer window, or repeated across several non-overlapping 30-day windows to check it isn't just regression to the mean for pages that happened to spike or dip once.

**Finding from the ML Appendix — "What Predicts Health?" (page 27).** A Random Forest predicts `health_score` from features including `avg_position`, `impressions`, and `scroll_depth`, and reports Average Position as the top predictor at 43% importance. The paper's own text flags this: "health score is partly constructed from inputs such as position and impressions" — so this is close to label leakage by construction, not just correlation. My methodology question: since the target is a formula that already contains two of the three top "predictors," does a holdout split even matter here? A model can score well on a holdout while still just be reading the label's own recipe back. I'd want a version that predicts health from features that are NOT inputs to the health-score formula (e.g. content age, word count, freshness) before trusting this as an insight about what actually drives performance, rather than a description of the formula.


In [1]:
# Section 1 is a reading exercise on the FlyRank paper (docs/flyrank-seo-research-march-2026.pdf),
# not a code check on my own data. No computation needed here.
print("See markdown above: 2 paper findings + my methodology question for each.")


See markdown above: 2 paper findings + my methodology question for each.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

My Week-5 notebook already used a client-grouped split (`GroupShuffleSplit` on `client_id`), so the "before" I'm testing here is what my numbers would have looked like if I had used the more common but dishonest choice: a plain random row split. Same model (logistic regression), same features, same seed — only the split changes.

The random split shares clients across train and test (31 of 32 clients appear on both sides), so the model can partly memorize per-client patterns instead of learning something that generalizes. The grouped split holds every client's rows entirely in one side, so client overlap is 0 and the score reflects performance on **clients the model never saw**, which is the honest question for this use case.


In [2]:
import pandas as pd, numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score

RANDOM_SEED = 42

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

window_cols = [c for c in df.columns if c.endswith("_last_30d") or c.endswith("_prev_30d")]
drop_cols = {"content_id", "client_id", "trend_direction", "trend_pct", "is_declining_label"} | set(window_cols)
feature_cols = [c for c in df.columns if c not in drop_cols]
num_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df[c])]
cat_cols = [c for c in feature_cols if not pd.api.types.is_numeric_dtype(df[c])]

def precision_at_k(y_true, scores, k):
    order = np.argsort(-scores)
    return float(np.asarray(y_true)[order][:k].mean())

def build_pipe():
    cat_pipe = Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="missing")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore"))])
    pre = ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num_cols),
        ("cat", cat_pipe, cat_cols),
    ])
    return Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_SEED))])

results = {}

# BEFORE: naive random row split — ignores that rows repeat by client
train_r, test_r = train_test_split(df, test_size=0.25, random_state=RANDOM_SEED, stratify=df["is_declining_label"])
model_r = build_pipe()
model_r.fit(train_r[feature_cols], train_r["is_declining_label"])
proba_r = model_r.predict_proba(test_r[feature_cols])[:, 1]
results["random_split (before)"] = {
    "p@10": precision_at_k(test_r["is_declining_label"].values, proba_r, 10),
    "p@20": precision_at_k(test_r["is_declining_label"].values, proba_r, 20),
    "roc_auc": roc_auc_score(test_r["is_declining_label"], proba_r),
    "client_overlap": len(set(train_r["client_id"]) & set(test_r["client_id"])),
}

# AFTER: grouped split by client_id — same approach used in Week-5
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))
train_g, test_g = df.iloc[train_idx], df.iloc[test_idx]
model_g = build_pipe()
model_g.fit(train_g[feature_cols], train_g["is_declining_label"])
proba_g = model_g.predict_proba(test_g[feature_cols])[:, 1]
results["grouped_split (after)"] = {
    "p@10": precision_at_k(test_g["is_declining_label"].values, proba_g, 10),
    "p@20": precision_at_k(test_g["is_declining_label"].values, proba_g, 20),
    "roc_auc": roc_auc_score(test_g["is_declining_label"], proba_g),
    "client_overlap": len(set(train_g["client_id"]) & set(test_g["client_id"])),
}

comparison = pd.DataFrame(results).T.round(3)
print("=== honest-split audit: random split vs client-grouped split ===")
comparison


=== honest-split audit: random split vs client-grouped split ===


,p@10,p@20,roc_auc,client_overlap
random_split (before),0.9,0.9,0.698,31.0
grouped_split (after),0.8,0.7,0.575,0.0


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Running the attack checklist from `hunting-leakage-and-validating/SKILL.md` against my final 34-column feature set:

- **Label-derived columns** (`trend_direction`, `trend_pct`) — excluded, these compute the label itself.
- **IDs** (`content_id`, `client_id`) — excluded from features, used only for grouping.
- **Overlapping-window columns** — `is_declining_label` is built from `impressions_last_30d` vs `impressions_prev_30d`, so I exclude every `*_last_30d` / `*_prev_30d` column; any of these in the feature set would just be the label wearing a different name.
- **Product/decision flags** — the starter CSV doesn't ship an existing system's flag column, so nothing to exclude here.
- **Correlation check** — no single numeric feature correlates with the label above 0.19 in absolute value, so there's no "one feature towers over the rest" red flag.
- **Confirmation the harness actually catches leakage** — I deliberately added the excluded window columns back in and reran the same grouped-split model. Precision@10 jumps from 0.80 to 1.00 and ROC AUC from 0.575 to 0.809, exactly the "collapse when removed" signature the skill describes for a label-derived leak. That confirms two things: the window columns really were leaking, and my test harness is sensitive enough to catch it — so their absence from the final feature set is doing real work, not just following a rule by habit.


In [3]:
# 3a. Confirm exactly what was excluded and why
print("Excluded — label-derived / IDs:", sorted(["content_id", "client_id", "trend_direction", "trend_pct"]))
print("Excluded — overlapping-window columns (share the label's own comparison window):")
print(window_cols)
print()

# 3b. No single feature should dominate — check correlation with the label
corrs = df[num_cols + ["is_declining_label"]].corr(numeric_only=True)["is_declining_label"].drop("is_declining_label")
corrs = corrs.reindex(corrs.abs().sort_values(ascending=False).index)
print("Top 8 numeric features by absolute correlation with the label:")
print(corrs.head(8).round(3))
print()

# 3c. Prove the harness catches leakage: add the excluded window columns back and re-score
leaky_feature_cols = [c for c in df.columns if c not in
                       {"content_id", "client_id", "trend_direction", "trend_pct", "is_declining_label"}]
leaky_num = [c for c in leaky_feature_cols if pd.api.types.is_numeric_dtype(df[c])]
leaky_cat = [c for c in leaky_feature_cols if not pd.api.types.is_numeric_dtype(df[c])]

cat_pipe2 = Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="missing")),
                       ("onehot", OneHotEncoder(handle_unknown="ignore"))])
pre_leaky = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), leaky_num),
    ("cat", cat_pipe2, leaky_cat),
])
model_leaky = Pipeline([("pre", pre_leaky), ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_SEED))])
model_leaky.fit(train_g[leaky_feature_cols], train_g["is_declining_label"])
proba_leaky = model_leaky.predict_proba(test_g[leaky_feature_cols])[:, 1]

print(f"WITHOUT window cols (final, clean):  p@10={comparison.loc['grouped_split (after)','p@10']:.3f}  "
      f"roc_auc={comparison.loc['grouped_split (after)','roc_auc']:.3f}")
print(f"WITH window cols added back (leak test): p@10={precision_at_k(test_g['is_declining_label'].values, proba_leaky, 10):.3f}  "
      f"roc_auc={roc_auc_score(test_g['is_declining_label'], proba_leaky):.3f}")


Excluded — label-derived / IDs: ['client_id', 'content_id', 'trend_direction', 'trend_pct']
Excluded — overlapping-window columns (share the label's own comparison window):
['impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']

Top 8 numeric features by absolute correlation with the label:
days_with_impressions     0.190
content_age_days         -0.164
age_tier_order           -0.156
word_count                0.090
days_since_last_update    0.081
char_count                0.072
ctr                      -0.062
clicks_90d               -0.040
Name: is_declining_label, dtype: float64



WITHOUT window cols (final, clean):  p@10=0.800  roc_auc=0.575
WITH window cols added back (leak test): p@10=1.000  roc_auc=0.809


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original (Week-5, Section 4):** "Both models beat the Week-4 rule baseline... so an observed model is a real step up from the hand-written rule for a top-of-queue reviewer... Logistic Regression is stronger at precision@10 (0.80) and precision@20 (0.70)... the simpler model is the better pick here despite the Random Forest's higher AUC."

This reads more certain than the evidence supports — it states a model comparison as settled fact from a single train/test split on one static CSV snapshot, with no confidence interval and no check across multiple seeds.

**Rewritten:** On this dataset and this single client-grouped split, logistic regression showed a **directionally higher observed precision@10 (0.80)** than random forest, and both **measured** above the 0.517 base rate and the Week-4 rule baseline (0.30). This is **decision-support** evidence for prioritizing logistic regression as the top-of-queue ranker in this internship pipeline — it is not a guarantee that the gap holds on a different data cut, a different seed, or in production, and it should be re-checked before being treated as a settled choice.


In [4]:
print("Original claim numbers (from Week-5, recomputed above under the grouped split):")
print(comparison.loc[["grouped_split (after)"]])
print()
print("Safe-language version uses: 'observed', 'measured', 'directional', 'decision-support' —")
print("see the rewritten paragraph in the markdown cell above.")


Original claim numbers (from Week-5, recomputed above under the grouped split):
                       p@10  p@20  roc_auc  client_overlap
grouped_split (after)   0.8   0.7    0.575             0.0

Safe-language version uses: 'observed', 'measured', 'directional', 'decision-support' —
see the rewritten paragraph in the markdown cell above.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.